In [8]:
%pip install -q "anthropic>=2.8.0,<3" pillow pendulum python-dotenv openai

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.2.17 requires numpy<2,>=1; python_version < "3.12", but you have numpy 2.0.2 which is incompatible.
langchain-community 0.2.16 requires numpy<2,>=1; python_version < "3.12", but you have numpy 2.0.2 which is incompatible.
google-genai 1.38.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.
deepeval 1.1.6 requires grpcio~=1.63.0, but you have grpcio 1.75.0 which is incompatible.
arxiv 2.1.3 requires requests~=2.32.0, but you have requests 2.34.2 which is incompatible.

[notice] A new release of pip is available: 23.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# 1. API claude zero-shot approach 

In [ ]:
import io
import os
from PIL import Image
import base64
from anthropic import Anthropic
import json
from pathlib import Path
import pendulum

def resize_image_under_limit(image_path, max_size_mb=4.5):
    
    max_size_bytes = max_size_mb * 1024 * 1024
    img = Image.open(image_path)

    
    img_format = img.format if img.format else 'JPEG'
    if img_format == 'MPO':
        img_format = 'JPEG'

  
    file_size = os.path.getsize(image_path)
    print(f"Dimensione originale dell'immagine: {file_size / (1024*1024):.2f} MB")

    if file_size <= max_size_bytes:
        print("L'immagine è già sotto il limite consentito.")
        with open(image_path, "rb") as f:
            return f.read(), img_format.lower()

    
    quality = 90
    scale_factor = 0.9

    while file_size > max_size_bytes and scale_factor > 0.1:
        width, height = img.size
        new_size = (int(width * scale_factor), int(height * scale_factor))
        temp_img = img.resize(new_size, Image.Resampling.LANCZOS)

        buffer = io.BytesIO()
        temp_img.save(buffer, format=img_format, quality=quality)
        data = buffer.getvalue()
        file_size = len(data)

        print(f"New dimension: {file_size / (1024*1024):.2f} MB (Scale: {scale_factor:.1f})")
        scale_factor -= 0.1
        quality = max(50, quality - 5)

    return data, img_format.lower()

In [ ]:

from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")
api_key = os.environ.get("ANTHROPIC_API_KEY")
if not api_key:
    raise RuntimeError("Set ANTHROPIC_API_KEY in .env .")
client = Anthropic(api_key=api_key)

def analyze_image_with_claude(image_path="", prompt="Please analyze the attached image and describe it according to the instructions. Return only the JSON array.", system_prompt="""
### ROLE
You are an expert in medieval diagram analysis. Your task is to analyse the diagram in the image in a single pass: classify the diagram by subject, describe the relationships between its elements on the basis of what is visible, and interpret them.
 
### GOAL
Your goal is to classify the diagram by subject (e.g. cosmological, computistical) and provide an interpretation.
 
### INTERPRETATION RULES
- Base every interpretation on the visible features and the transcribed text first; use external knowledge only to complete them. State in "basis" what the interpretation mainly relies on.
- Do not invent sources. Mention an author, text or tradition only if you are confident of it; otherwise give the interpretation without naming a source and lower "interpretation_certainty".
- If an element or relation cannot be interpreted, write "uninterpreted" in the corresponding field and set "interpretation_certainty" to "low".
- When the same element appears in several entries, give it the same interpretation every time.
- Do not add external knowledge to the interpretation; rely only on what is visible and transcribed from the diagram.
### OUTPUT
Return only a valid JSON object, with no explanation, no additional text, and no markdown code fences. Any double quote inside a string value must be escaped (\").
 
### OUTPUT FORMAT
[
  {
    "id": 1,
    "diagram_classification": "",
    "interpretation": "",
    "basis": "textual",
    "certainty": "high",
    "evidence": "",
    "updated_at": ""
  }
]
 
### FIELD RULES
- "id", "subject", "predicate", "object": copied unchanged from Step 1.
- "classification": the category of the diagram (e.g. cosmological, theological...).
- "interpretation": interpretation of the meaning of the diagram.
- "basis": the main ground of the interpretation, exactly one of "visual" (visible features), "textual" (transcribed text in the diagram).
- "certainty": your confidence in the interpretation of that entry: "high", "medium", or "low".
- "evidence": the reasons supporting the interpretation.
- "updated_at": always an empty string; it will be filled in by a Python library.
"""):
    
    image_bytes, img_type = resize_image_under_limit(image_path)

   
    media_type = f"image/{'jpeg' if img_type == 'jpg' else img_type}"

    
    base64_image = base64.b64encode(image_bytes).decode('utf-8')

   
    message = client.messages.create(
        model="claude-sonnet-5-5",
        max_tokens=8192,
        system=system_prompt,  
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": media_type,
                            "data": base64_image,
                        },
                    },
                    {
                        "type": "text",
                        "text": prompt
                    }
                ],
            }
        ],
    )
    text_blocks = [block.text for block in message.content if block.type == "text"]
    if not text_blocks:
        raise ValueError("Claude non ha restituito blocchi di testo.")
    return "\n".join(text_blocks)



project_dir = Path(r"")
images_dir = ""
image_extensions = {".jpg", ".jpeg", ".png", ".gif", ".webp"}
image_paths = sorted(
    p for p in images_dir.rglob("*") if p.is_file() and p.suffix.lower() in image_extensions
)
print(f"Found {len(image_paths)} images")

for image_path in image_paths:
   
    output_path = image_path.with_name(image_path.stem + "").with_suffix(".json")
    try:
        if output_path.exists():
            print(f"Skipped {image_path}: {output_path} already exists")
            continue
        response = analyze_image_with_claude(str(image_path))
        print(f"Response from Claude for {image_path}: {response}")
        data = json.loads(response)
        if not isinstance(data, list):
            raise ValueError("Expected a JSON array from Claude.")
        if not all(isinstance(item, dict) for item in data):
            raise ValueError("Expected every JSON array entry to be an object.")

        updated_at = pendulum.now("UTC").format("YYYY-MM-DDTHH:mm[Z]")
        for item in data:
            item["updated_at"] = updated_at

        with output_path.open("w", encoding="utf-8") as file:
            json.dump(data, file, ensure_ascii=False, indent=2)
        print(f"Saved to: {output_path}")
    except Exception as e:
        print(f"Failed on {image_path}: {e}")

Found 13 images
Dimensione originale dell'immagine: 0.06 MB
L'immagine è già sotto il limite consentito.
Response from Claude for C:\Users\crosi\Documents\GitHub\DH27\corpus_Investigation (2)\images\0953127b-451f-425c-be99-f138efdc7ab9\0953127b-451f-425c-be99-f138efdc7ab9.jpg: [
  {
    "id": 1,
    "diagram_classification": "theological (Trinitarian diagram, a form of the Scutum Fidei or Shield of the Trinity)",
    "interpretation": "The diagram sets out the doctrine of the Trinity. Three outer roundels, PATER (Father), SPIRITUS SANCTUS (Holy Spirit, abbreviated and hard to read) and FILIUS (Son), sit at the corners of an inverted triangle. Each is joined to the central roundel DEUS (God) by a band labelled EST (\"is\"). The three outer sides of the triangle, which link the Persons to one another, are labelled NON EST (\"is not\"). The diagram therefore says that the Father is God, the Son is God and the Holy Spirit is God, while the Father is not the Son, the Son is not the Holy Spi

# 2. Chain of Thought (CoT)

In [ ]:

class ClaudeChat:
    def __init__(self, system_prompt=""):
        import os
        from inspect import signature
        from pathlib import Path
        from anthropic import Anthropic
        from dotenv import load_dotenv

        load_dotenv(Path.cwd() / ".env")
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise RuntimeError("Set ANTHROPIC_API_KEY in .env.")

        self.client = Anthropic(api_key=api_key)
        if "cache_control" not in signature(self.client.messages.create).parameters:
            raise RuntimeError("Update Anthropic and restart the kernel")
        self.system_prompt = system_prompt
        self.messages = []

    def start_chat_with_image(self, image_path, prompt="Please analyze the attached image and describe it according to the instructions. Return only the JSON array."):
        
        image_bytes, img_type = resize_image_under_limit(image_path)
        media_type = f"image/{'jpeg' if img_type == 'jpg' else img_type}"
        base64_image = base64.b64encode(image_bytes).decode('utf-8')

       
        self.messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": media_type,
                            "data": base64_image,
                        },
                    },
                    {
                        "type": "text",
                        "text": prompt
                    }
                ],
            }
        ]

        return self._send_request()

    def send_message(self, prompt):
        
        if not self.messages:
            raise ValueError("First conversation start with 'start_chat_with_image'.")

       
        self.messages.append({
            "role": "user",
            "content": [{"type": "text", "text": prompt}]
        })

        return self._send_request()

    def _send_request(self):
        
        response = self.client.messages.create(
            model="claude-sonnet-5-5",
            max_tokens=20000,
            system=self.system_prompt,
            cache_control={"type": "ephemeral"},
            messages=self.messages
        )
        self.last_usage = response.usage

        assistant_response = "".join(
            block.text for block in response.content
            if getattr(block, "type", None) == "text" and block.text
        )
        if not assistant_response:
            tipi = [getattr(b, "type", None) for b in response.content]
            raise RuntimeError(f"No text found in response (stop_reason={response.stop_reason}, blocchi={tipi})")

        
        self.messages.append({
            "role": "assistant",
            "content": [{"type": "text", "text": assistant_response}]
        })

        return assistant_response

In [ ]:
basic_prompt = (Path.cwd() / "systemprompt_general.md").read_text(encoding="utf-8")
interpretation_prompt = (Path.cwd() / "userprompt_interpretation.md").read_text(encoding="utf-8")
element_prompt = (Path.cwd() / "userprompt_basic.md").read_text(encoding="utf-8")

project_dir = Path(r"")
images_dir = project_dir / "masking"
image_extensions = {".jpg", ".jpeg", ".png", ".gif", ".webp"}
image_paths = sorted(
    p for p in images_dir.rglob("*") if p.is_file() and p.suffix.lower() in image_extensions
)
print(f"Found {len(image_paths)} images")

for image_path in image_paths:
    output_path = image_path.with_name(image_path.stem + "")
    try:
        if output_path.exists():
            print(f"Skipped {image_path}: {output_path} already exists")
            continue

        chat_session = ClaudeChat(system_prompt=basic_prompt)
        print(f"--- Round 1 (sending image): {image_path} ---")
        risposta_1 = chat_session.start_chat_with_image(str(image_path), prompt=element_prompt)
        print(f"Claude: {risposta_1}\n")
        element_analysis = json.loads(risposta_1)
        if not isinstance(element_analysis, list) or not all(isinstance(item, dict) for item in element_analysis):
            raise ValueError("First round outputs a JSON")
        print(f"Token in cache: {chat_session.last_usage.cache_creation_input_tokens}")

        print("--- Round 2 (only text, image is in context) ---")
        risposta_2 = chat_session.send_message(interpretation_prompt)
        print(f"Claude: {risposta_2}")
        interpretation = json.loads(risposta_2)
        if not isinstance(interpretation, list) or not all(isinstance(item, dict) for item in interpretation):
            raise ValueError("Second round outputs a JSON")
        print(f"Token in cache: {chat_session.last_usage.cache_read_input_tokens}")

        with output_path.open("w", encoding="utf-8") as file:
            json.dump({"element_analysis": element_analysis, "interpretation": interpretation}, file, ensure_ascii=False, indent=2)
        print(f"Saved in: {output_path}")
    except Exception as e:
        print(f"Error for {image_path} ({type(e).__name__}): {e}")

Found 12 images
--- Turno 1 (Invio immagine): C:\Users\crosi\Documents\GitHub\DH27\masking\0953127b-451f-425c-be99-f138efdc7ab9.JPG ---
Dimensione originale dell'immagine: 0.06 MB
L'immagine è già sotto il limite consentito.
Claude: [
  {
    "id": 1,
    "relation_type": "geometric",
    "subject": "upper left circle",
    "predicate": "connected_to",
    "object": "upper right circle",
    "certainty": "high",
    "evidence": "Connected by a straight double-line band running horizontally along the top of the inverted triangle, from the upper left circle to the upper right circle.",
    "updated_at": ""
  },
  {
    "id": 2,
    "relation_type": "geometric",
    "subject": "upper left circle",
    "predicate": "connected_to",
    "object": "central circle",
    "certainty": "high",
    "evidence": "Connected by a straight double-line band running diagonally downward to the right from the upper left circle to the central circle.",
    "updated_at": ""
  },
  {
    "id": 3,
    "relatio